# LeetCode 2229 — Check if an Array Is Consecutive bằng Python

> **Bài Premium:** đề viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.

Link đề: https://leetcode.com/problems/check-if-an-array-is-consecutive/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Sorting

Skeleton LeetCode:

```python
class Solution:
    def isConsecutive(self, nums: List[int]) -> bool:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu định nghĩa mảng **liên tiếp** (consecutive): chứa **đủ mọi số** trong đoạn `[x, x + n - 1]`, với `x` là phần tử nhỏ nhất.
2. Dùng `set`, `sorted`, `min`, `max`, `len` thành thạo, và biết `set` loại bỏ phần tử trùng.
3. Viết 4 cách: brute force (tìm từng số), sắp xếp rồi so hiệu liền kề, `set` + công thức `max - min`, và mảng đánh dấu (bucket).
4. Rút ra **điều kiện cần và đủ**: `không trùng` và `max - min == n - 1` (nguyên lý chuồng bồ câu).
5. Tránh lỗi hay gặp: chỉ kiểm tra `max - min == n - 1` mà quên trùng lặp, dùng tổng thay cho kiểm tra thật, quên mảng 1 phần tử.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Mảng được gọi là **liên tiếp** nếu nó chứa **mọi số** trong khoảng `[x, x + n - 1]` (bao gồm hai đầu), trong đó `x` là số nhỏ nhất của mảng và `n` là độ dài mảng. Trả về `true` nếu `nums` liên tiếp, ngược lại `false`.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[1,3,4,2]` | `true` | `x = 1`, `n = 4`, cần đủ `1,2,3,4` — có đủ. |
| 2 | `[1,3]` | `false` | `x = 1`, `n = 2`, cần `1,2` nhưng không có `2`. |
| 3 | `[3,5,4]` | `true` | `x = 3`, `n = 3`, cần `3,4,5` — có đủ. |

Constraints:

- `1 <= nums.length <= 10^5`
- `0 <= nums[i] <= 10^9`

Lưu ý: thứ tự các phần tử **không quan trọng** (ví dụ 1 và 3 đều không sắp xếp sẵn). Mảng có số trùng như `[2,2,3]` **không** liên tiếp, vì `n = 3` cần `2,3,4` mà không có `4`.

In [1]:
examples = [([1, 3, 4, 2], True), ([1, 3], False), ([3, 5, 4], True)]
for nums, expected in examples:
    x, n = min(nums), len(nums)
    print("nums =", nums, "| cần có:", list(range(x, x + n)), "| mong đợi =", expected)

nums = [1, 3, 4, 2] | cần có: [1, 2, 3, 4] | mong đợi = True
nums = [1, 3] | cần có: [1, 2] | mong đợi = False
nums = [3, 5, 4] | cần có: [3, 4, 5] | mong đợi = True


## 3. Kiến thức nền

### 3.1. `min`, `max`, `len`

`min(nums)` và `max(nums)` cho phần tử nhỏ/lớn nhất, mỗi lần duyệt tốn `O(n)`. `len(nums)` cho số phần tử, tốn `O(1)`.

In [2]:
nums = [3, 5, 4]
print(min(nums), max(nums), len(nums))
print(max(nums) - min(nums) + 1)        # số giá trị trong đoạn [min, max]

3 5 3
3


### 3.2. `set` — tập hợp không trùng, tra cứu nhanh

`set(nums)` bỏ trùng; `len(set(nums))` là số giá trị **khác nhau**. Kiểm tra `x in s` trung bình `O(1)`, trong khi `x in list` tốn `O(n)`.

In [3]:
nums = [2, 2, 3, 5]
s = set(nums)
print(s, len(nums), len(s))
print(3 in s, 4 in s)
print(len(nums) == len(s))       # True nếu không có số trùng

{2, 3, 5} 4 3
True False
False


### 3.3. `range(x, x + n)` và `all(...)`

`range(x, x + n)` sinh `x, x+1, ..., x+n-1` (đúng `n` số). `all(cond for ...)` trả `True` nếu **mọi** điều kiện đúng và dừng sớm ngay khi gặp một điều kiện sai.

In [4]:
x, n = 3, 3
print(list(range(x, x + n)))
print(all(v in {3, 4, 5} for v in range(x, x + n)))
print(all(v in {3, 5} for v in range(x, x + n)))

[3, 4, 5]
True
False


### 3.4. `sorted` và so sánh phần tử liền kề

`sorted(nums)` trả danh sách mới đã tăng dần. Sau sắp xếp, mảng liên tiếp có **hiệu mỗi cặp liền kề đúng bằng `1`**. `zip(a, a[1:])` cho các cặp liền kề.

In [5]:
nums = [1, 3, 4, 2]
a = sorted(nums)
print(a)
print([y - x for x, y in zip(a, a[1:])])
print(all(y - x == 1 for x, y in zip(a, a[1:])))

[1, 2, 3, 4]
[1, 1, 1]
True


### 3.5. Nguyên lý chuồng bồ câu (pigeonhole)

Có `n` số **đôi một khác nhau** nằm trong đoạn `[min, max]` có `max - min + 1` giá trị. Nếu `max - min + 1 == n` thì `n` số phải **lấp đầy** từng giá trị của đoạn (mỗi "ngăn" đúng một số). Đây là chìa khóa của cách tối ưu.

In [6]:
nums = [4, 6, 5, 7]
print("max - min + 1 =", max(nums) - min(nums) + 1, "| n =", len(nums), "| khác nhau:", len(set(nums)))
nums = [4, 6, 6, 7]       # có trùng
print("max - min + 1 =", max(nums) - min(nums) + 1, "| n =", len(nums), "| khác nhau:", len(set(nums)))

max - min + 1 = 4 | n = 4 | khác nhau: 4
max - min + 1 = 4 | n = 4 | khác nhau: 3


## 4. Ý tưởng nền tảng — tính bằng tay

Lấy `nums = [1,3,4,2]`: `x = 1`, `n = 4`. Ta cần `1,2,3,4`.

| Số cần có | Có trong `nums`? |
|---|---|
| 1 | có |
| 2 | có |
| 3 | có |
| 4 | có |

Đủ cả ⇒ `true`.

Lấy `nums = [1,3]`: `x = 1`, `n = 2`, cần `1,2`. Không có `2` ⇒ `false`.

Lấy `nums = [2,2,3]`: `x = 2`, `n = 3`, cần `2,3,4`. Không có `4` ⇒ `false` (số `2` bị lặp nên "chiếm chỗ" của `4`).

Tóm lại: bài hỏi **"n vị trí liên tiếp từ min có được lấp đầy không?"**.

In [7]:
for nums in ([1, 3, 4, 2], [1, 3], [2, 2, 3]):
    x, n = min(nums), len(nums)
    need = list(range(x, x + n))
    missing = [v for v in need if v not in nums]
    print(nums, "cần", need, "| thiếu", missing, "->", not missing)

[1, 3, 4, 2] cần [1, 2, 3, 4] | thiếu [] -> True
[1, 3] cần [1, 2] | thiếu [2] -> False
[2, 2, 3] cần [2, 3, 4] | thiếu [4] -> False


## 5. Cách 1 — Brute Force: tìm từng số cần có

Đúng theo đề: lấy `x = min(nums)`, rồi với mỗi `v` từ `x` tới `x + n - 1`, kiểm tra `v in nums` (tìm tuần tự trong list). Thiếu bất kỳ số nào thì trả `False`.

In [8]:
class SolutionBrute:
    def isConsecutive(self, nums):
        x, n = min(nums), len(nums)
        for v in range(x, x + n):
            if v not in nums:           # tìm tuần tự O(n)
                return False
        return True

sol = SolutionBrute()
print(sol.isConsecutive([1, 3, 4, 2]))
print(sol.isConsecutive([1, 3]))
print(sol.isConsecutive([3, 5, 4]))

True
False
True


**Độ phức tạp:** `n` lần tra cứu, mỗi lần `O(n)` ⇒ Time `O(n²)`, Space `O(1)`. Với `n = 10^5` là `10^10` thao tác — **quá chậm** (TLE).

**Nút thắt:** `v in nums` trên list là tìm tuần tự. Nếu đổi `nums` thành `set` thì tra cứu `O(1)`, hoặc dùng sắp xếp để khỏi tra cứu.

## 6. Ý tưởng thuật toán — bước đột phá

Ta không cần kiểm tra từng số. Chỉ cần **hai sự thật**:

1. Mọi số **đôi một khác nhau** (`len(set(nums)) == n`).
2. Khoảng trải **vừa khít**: `max - min == n - 1`.

Vì sao hai điều kiện này đủ? `n` số khác nhau trong đoạn `[min, max]` chỉ có `max - min + 1` ô. Nếu `max - min + 1 == n` thì `n` số phải chiếm **hết** các ô ⇒ đủ mọi giá trị. Ngược lại, nếu mảng liên tiếp thì hiển nhiên không trùng và `max = min + n - 1`.

Quy tắc "nếu… thì…":

- **Nếu** có số trùng **thì** `false` (vì `n` số mà ít hơn `n` giá trị khác nhau, không thể phủ `n` giá trị).
- **Nếu** `max - min != n - 1` **thì** `false`.
- **Còn lại** `true`.

## 7. Các cách tiếp theo

### Cách 2 — Sắp xếp rồi kiểm tra hiệu liền kề

Sau `sorted`, mảng liên tiếp phải có mọi hiệu `a[i+1] - a[i] == 1`. Một số trùng cho hiệu `0`, một chỗ hổng cho hiệu `> 1` ⇒ đều bị loại.

In [9]:
class SolutionSort:
    def isConsecutive(self, nums):
        a = sorted(nums)
        for i in range(len(a) - 1):
            if a[i + 1] - a[i] != 1:      # trùng (0) hoặc hổng (>1)
                return False
        return True

sol = SolutionSort()
print(sol.isConsecutive([1, 3, 4, 2]))
print(sol.isConsecutive([1, 3]))
print(sol.isConsecutive([3, 5, 4]))

True
False
True


**Độ phức tạp:** Time `O(n log n)` do sắp xếp, Space `O(n)` (bản sao đã sắp xếp; `O(1)` nếu sort tại chỗ).

### Cách 3 — `set` + công thức `max - min == n - 1`

Một lượt tính `min`, `max`, và `set`. Không cần sắp xếp.

In [10]:
class SolutionSet:
    def isConsecutive(self, nums):
        n = len(nums)
        if len(set(nums)) != n:           # có số trùng
            return False
        return max(nums) - min(nums) == n - 1

sol = SolutionSet()
print(sol.isConsecutive([1, 3, 4, 2]))
print(sol.isConsecutive([1, 3]))
print(sol.isConsecutive([3, 5, 4]))

True
False
True


**Độ phức tạp:** Time `O(n)`, Space `O(n)` cho `set`.

### Cách 4 — Mảng đánh dấu (bucket) không dùng hash

Giá trị tới `10^9` nên không thể dùng mảng theo giá trị, nhưng sau khi trừ `min`, một số hợp lệ phải nằm trong `[0, n - 1]`. Ta tạo mảng `seen` kích thước `n`: nếu `nums[i] - min >= n` ⇒ `false`; nếu ô đã đánh dấu ⇒ trùng ⇒ `false`.

In [11]:
class SolutionBucket:
    def isConsecutive(self, nums):
        n, lo = len(nums), min(nums)
        seen = [False] * n
        for v in nums:
            idx = v - lo                  # vị trí tương đối so với min
            if idx >= n or seen[idx]:     # ngoài khoảng hoặc trùng
                return False
            seen[idx] = True
        return True

sol = SolutionBucket()
print(sol.isConsecutive([1, 3, 4, 2]))
print(sol.isConsecutive([1, 3]))
print(sol.isConsecutive([3, 5, 4]))

True
False
True


**Độ phức tạp:** Time `O(n)`, Space `O(n)` cho `seen`. Ưu điểm: không phụ thuộc hash, dừng sớm khi gặp trùng hoặc giá trị vượt khoảng. Đây cũng là bước đệm cho bài "First Missing Positive".

## 8. TRACE từng bước

Trace cách 4 với `nums = [3,5,4]` (`n = 3`, `lo = 3`):

| v | idx = v - lo | `idx >= n`? | `seen[idx]` trước | seen sau |
|---|---|---|---|---|
| 3 | 0 | không | False | `[T,F,F]` |
| 5 | 2 | không | False | `[T,F,T]` |
| 4 | 1 | không | False | `[T,T,T]` |

Hết mảng mà không vi phạm ⇒ `True`. Hàm trace dưới đây in từng bước, kèm một ví dụ trả `False` (`[2,2,3]`).

In [12]:
def trace_bucket(nums):
    n, lo = len(nums), min(nums)
    seen = [False] * n
    print(f"n={n}, lo={lo}")
    for v in nums:
        idx = v - lo
        if idx >= n:
            print(f"v={v} idx={idx} >= n -> ngoài khoảng, False")
            return False
        if seen[idx]:
            print(f"v={v} idx={idx} đã đánh dấu -> trùng, False")
            return False
        seen[idx] = True
        print(f"v={v} idx={idx} -> seen={seen}")
    print("đủ mọi ô -> True")
    return True

trace_bucket([3, 5, 4])

n=3, lo=3
v=3 idx=0 -> seen=[True, False, False]
v=5 idx=2 -> seen=[True, False, True]
v=4 idx=1 -> seen=[True, True, True]
đủ mọi ô -> True


True

In [13]:
trace_bucket([2, 2, 3])
trace_bucket([1, 3])

n=3, lo=2
v=2 idx=0 -> seen=[True, False, False]
v=2 idx=0 đã đánh dấu -> trùng, False
n=2, lo=1
v=1 idx=0 -> seen=[True, False]
v=3 idx=2 >= n -> ngoài khoảng, False


False

## 9. Code nộp LeetCode

Dùng cách 3 (`set` + `max - min`): ngắn, `O(n)`.

In [14]:
from typing import List

class Solution:
    def isConsecutive(self, nums: List[int]) -> bool:
        n = len(nums)
        if len(set(nums)) != n:
            return False
        return max(nums) - min(nums) == n - 1

print(Solution().isConsecutive([1, 3, 4, 2]))
print(Solution().isConsecutive([1, 3]))
print(Solution().isConsecutive([3, 5, 4]))

True
False
True


## 10. Vì sao thuật toán đúng?

**Chiều thuận (mảng liên tiếp ⇒ hai điều kiện):** mảng chứa đúng `n` giá trị `x, x+1, …, x+n-1` — mỗi giá trị đúng một lần vì độ dài là `n` và có đủ `n` giá trị khác nhau. Vậy không trùng và `max - min = n - 1`.

**Chiều ngược (hai điều kiện ⇒ liên tiếp):** `n` số đôi một khác nhau, đều thuộc `[min, max]`. Đoạn này có đúng `max - min + 1 = n` số nguyên. Một ánh xạ đơn ánh từ `n` phần tử vào tập `n` giá trị phải là toàn ánh, nên mỗi giá trị của đoạn được dùng đúng một lần ⇒ mảng chứa đủ `[min, min+n-1]`, nghĩa là liên tiếp.

**Cách sắp xếp:** nếu mọi hiệu liền kề bằng `1` thì dãy là `a0, a0+1, …, a0+n-1` — rõ ràng đủ. Ngược lại có hiệu `0` (trùng) hoặc `>=2` (hổng) ⇒ thiếu giá trị.

**Cách bucket:** mỗi số hợp lệ phải rơi vào một ô riêng trong `[0, n-1]`; vi phạm là trùng hoặc vượt khoảng, đúng như hai điều kiện trên.

**Một ví dụ nhỏ để thấy chuồng bồ câu:** `nums = [4, 6, 5]`, `min = 4`, `max = 6`, đoạn `[4, 6]` có `3` ô và có `3` số khác nhau ⇒ lấp đầy. Còn `nums = [4, 6, 6]` có `3` số nhưng chỉ `2` giá trị khác nhau, nên chắc chắn còn một ô trống (ô `5`) — không thể liên tiếp, bất kể `max - min` bằng bao nhiêu.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(n²)` | `O(1)` |
| 2. Sắp xếp | `O(n log n)` | `O(n)` (hoặc `O(1)` nếu sort tại chỗ) |
| 3. `set` + max/min | `O(n)` | `O(n)` |
| 4. Bucket | `O(n)` | `O(n)` |

Vì `n <= 10^5`, brute force **TLE**; ba cách còn lại đều qua. Cách 3 ngắn nhất, cách 4 không cần hash.

## 12. Bộ test

Test ví dụ đề, edge case, rồi đối chiếu 5 lớp trên input ngẫu nhiên có `seed`.

In [15]:
solutions = [SolutionBrute(), SolutionSort(), SolutionSet(), SolutionBucket(), Solution()]

for s in solutions:
    assert s.isConsecutive([1, 3, 4, 2]) is True
    assert s.isConsecutive([1, 3]) is False
    assert s.isConsecutive([3, 5, 4]) is True
print("ví dụ đề: OK")

ví dụ đề: OK


In [16]:
for s in solutions:
    assert s.isConsecutive([7]) is True                       # một phần tử luôn liên tiếp
    assert s.isConsecutive([0]) is True
    assert s.isConsecutive([5, 5]) is False                   # trùng
    assert s.isConsecutive([2, 2, 3]) is False                # trùng và thiếu
    assert s.isConsecutive([1, 1, 4]) is False                # tổng "đẹp" nhưng sai (phản ví dụ của cách dùng tổng)
    assert s.isConsecutive([10**9, 10**9 - 1]) is True        # giá trị lớn
    assert s.isConsecutive([0, 2, 1, 3, 5, 4]) is True        # bắt đầu từ 0, xáo trộn
    assert s.isConsecutive([1, 2, 4, 5]) is False             # hổng ở giữa
print("edge case: OK")

edge case: OK


In [17]:
import random
random.seed(2229)
yes = no = 0
for _ in range(1500):
    n = random.randint(1, 8)
    if random.random() < 0.5:
        x = random.randint(0, 20)
        nums = list(range(x, x + n))
        random.shuffle(nums)                                  # chắc chắn liên tiếp
    else:
        nums = [random.randint(0, 10) for _ in range(n)]      # thường không liên tiếp
    results = [s.isConsecutive(nums) for s in solutions]
    assert len(set(results)) == 1, (nums, results)
    yes += results[0]; no += not results[0]
print("random đối chiếu: True =", yes, "| False =", no, "| mọi cách khớp")

random đối chiếu: True = 877 | False = 623 | mọi cách khớp


### Kiểm chứng vét cạn (exhaustive) cho mảng nhỏ

Ngoài test ngẫu nhiên, với mảng nhỏ ta có thể **liệt kê mọi khả năng**: mọi mảng độ dài từ `1` đến `5` có giá trị trong `0..6`. Dùng `itertools.product` sinh toàn bộ, rồi so sánh lời giải nộp với định nghĩa trong đề (tức là `set(range(x, x+n)) == set(nums)` với `x = min`). Đây là cách mạnh nhất để chắc rằng công thức `không trùng + max - min == n - 1` không bỏ sót trường hợp nào.

In [18]:
from itertools import product

def by_definition(nums):
    x, n = min(nums), len(nums)
    return set(range(x, x + n)) == set(nums) and len(set(nums)) == n

total = 0
for n in range(1, 6):
    for nums in product(range(7), repeat=n):
        nums = list(nums)
        assert Solution().isConsecutive(nums) == by_definition(nums), nums
        assert SolutionBucket().isConsecutive(nums) == by_definition(nums), nums
        total += 1
print("đã kiểm tra vét cạn", total, "mảng, tất cả khớp định nghĩa")

đã kiểm tra vét cạn 19607 mảng, tất cả khớp định nghĩa


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ kiểm tra `max - min == n - 1`, quên số trùng

`[1,1,3]`: `max - min = 2 = n - 1` nhưng không có `2`.

In [19]:
def wrong_no_dup(nums):
    return max(nums) - min(nums) == len(nums) - 1

print("sai :", wrong_no_dup([1, 1, 3]))          # True (sai)
print("đúng:", Solution().isConsecutive([1, 1, 3]))

sai : True
đúng: False


### Lỗi 2 — Dùng tổng để "kiểm tra" thay cho kiểm tra thật

Tổng của `n` số liên tiếp từ `x` là `n*x + n*(n-1)/2`. Nhưng nhiều mảng khác cũng có cùng tổng — ví dụ `[1,1,4]` và `[1,2,3]` đều có tổng `6`.

In [20]:
def wrong_sum(nums):
    n, x = len(nums), min(nums)
    return sum(nums) == n * x + n * (n - 1) // 2

print("sai :", wrong_sum([1, 1, 4]))             # True (sai)
print("đúng:", Solution().isConsecutive([1, 1, 4]))

sai : True
đúng: False


### Lỗi 3 — Sắp xếp rồi chỉ so `a[-1] - a[0]`

Cách này chính là lỗi 1 trá hình. Phải so **từng** cặp liền kề, hoặc kiểm tra thêm trùng.

In [21]:
def wrong_sorted_ends(nums):
    a = sorted(nums)
    return a[-1] - a[0] == len(a) - 1

print("sai :", wrong_sorted_ends([2, 4, 4]))     # 4 - 2 = 2 = n - 1 -> True (sai)
print("đúng:", Solution().isConsecutive([2, 4, 4]))

sai : True
đúng: False


### Lỗi 4 — Dùng `v in nums` trên list cho `n = 10^5`

Cách brute force đúng kết quả nhưng `O(n²)` nên quá thời gian. Hãy đổi sang `set` để tra cứu `O(1)`.

In [22]:
import time

big = list(range(1, 3001))
random.shuffle(big)
t0 = time.perf_counter(); SolutionBrute().isConsecutive(big); t1 = time.perf_counter()
Solution().isConsecutive(big); t2 = time.perf_counter()
print(f"brute (list): {t1 - t0:.4f}s | tối ưu (set): {t2 - t1:.6f}s -> brute chậm hơn rất nhiều khi n lớn")

brute (list): 0.0307s | tối ưu (set): 0.000264s -> brute chậm hơn rất nhiều khi n lớn


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chứa mọi số trong một đoạn liên tiếp" ⇒ nghĩ ngay tới **`set`**, hoặc điều kiện `max - min == n - 1` **kèm không trùng**.
- Điều kiện cần-đủ: *khác nhau đôi một* + *khoảng trải vừa khít* (nguyên lý chuồng bồ câu).
- Có thể dùng mảng `seen` kích thước `n` sau khi trừ `min` để tránh hash.
- Đừng tin tổng hoặc chỉ `max - min`; luôn nghĩ tới phản ví dụ có số trùng.
- Bài họ hàng: Missing Number (268), First Missing Positive (41), Longest Consecutive Sequence (128), Contains Duplicate (217), Find All Numbers Disappeared in an Array (448).

## 15. Biến thể của bài

1. **Liệt kê các số bị thiếu** trong đoạn `[min, max]`.
2. **Tối thiểu số phần tử cần thêm** để mảng trở nên liên tiếp: `(max - min + 1) - số giá trị khác nhau`.
3. **Chuỗi liên tiếp dài nhất (LC 128):** duyệt từng điểm bắt đầu `v` mà `v - 1` không có trong `set`.
4. **Liên tiếp theo bước `d`:** các số lập thành `x, x+d, x+2d, …` (cấp số cộng).
5. **Liên tiếp vòng:** coi các số theo modulo `m`.

In [23]:
# Biến thể 1 & 2: số bị thiếu và số phần tử cần thêm
def missing_numbers(nums):
    s = set(nums)
    return [v for v in range(min(nums), max(nums) + 1) if v not in s]

nums = [1, 4, 6, 4]
print(missing_numbers(nums), len(missing_numbers(nums)))

[2, 3, 5] 3


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | Với mỗi giá trị cần có, tìm trong list | `O(n²)` | `O(1)` | Chỉ để hiểu đề |
| 2. Sắp xếp | Hiệu liền kề phải bằng `1` | `O(n log n)` | `O(n)` | Dễ nghĩ, không cần hash |
| 3. `set` + max/min | Không trùng và `max - min == n - 1` | `O(n)` | `O(n)` | **Nộp bài**, ngắn gọn |
| 4. Bucket | Mảng `seen` kích thước `n` theo `v - min` | `O(n)` | `O(n)` | Tránh hash, nền cho First Missing Positive |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `Solution().isConsecutive([5, 4, 6, 7])` là gì?

**Bài 2 (tính tay).** Với `nums = [10, 12, 11, 14]` hãy liệt kê các số cần có và số bị thiếu.

**Bài 3 (điền TODO).** Hoàn thành `has_duplicate(nums)` dùng `set`.

**Bài 4 (tự viết lại).** Viết `is_consecutive_sorted(nums)` bằng cách sắp xếp rồi kiểm tra hiệu liền kề (không nhìn code mẫu).

**Bài 5 (mini challenge).** Viết `min_additions(nums)`: số phần tử tối thiểu cần **thêm** vào để mảng trở nên liên tiếp (mảng có thể có trùng; số trùng không giúp ích gì).

In [24]:
# Bài 1: đoán rồi chạy
print(Solution().isConsecutive([5, 4, 6, 7]))

True


In [25]:
# Bài 3, 4, 5: TODO - KHÔNG được gọi khi run all
def has_duplicate(nums):
    # TODO: trả True nếu nums có phần tử lặp
    pass

def is_consecutive_sorted(nums):
    # TODO: sắp xếp rồi kiểm tra mọi hiệu liền kề == 1
    pass

def min_additions(nums):
    # TODO: số phần tử cần thêm để mảng liên tiếp
    pass

## 18. Gợi ý 3 mức

**Bài 3**
- Mức 1: `set` không giữ phần tử trùng.
- Mức 2: So sánh `len(set(nums))` với `len(nums)`.
- Mức 3: `return len(set(nums)) != len(nums)`.

**Bài 4**
- Mức 1: Sau `sorted`, hai phần tử liền kề cần hơn kém nhau đúng `1`.
- Mức 2: Dùng `zip(a, a[1:])`.
- Mức 3: `return all(y - x == 1 for x, y in zip(a, a[1:]))`.

**Bài 5**
- Mức 1: Số ô trong đoạn `[min, max]` là `max - min + 1`.
- Mức 2: Số ô đã có giá trị là `len(set(nums))`.
- Mức 3: `return (max(nums) - min(nums) + 1) - len(set(nums))`.

## 19. Đáp án tham khảo cho bài tập

Mỗi đáp án được kiểm tra bằng `assert`.

In [26]:
# Bài 1: 4,5,6,7 -> liên tiếp
assert Solution().isConsecutive([5, 4, 6, 7]) is True

# Bài 2: cần 10,11,12,13 -> thiếu 13 (và có 14 thừa); n = 4 nên cần [10..13]
need = list(range(min([10, 12, 11, 14]), min([10, 12, 11, 14]) + 4))
assert need == [10, 11, 12, 13]
assert [v for v in need if v not in [10, 12, 11, 14]] == [13]
assert Solution().isConsecutive([10, 12, 11, 14]) is False

# Bài 3
def has_duplicate_ans(nums):
    return len(set(nums)) != len(nums)
assert has_duplicate_ans([1, 2, 1]) is True
assert has_duplicate_ans([1, 2, 3]) is False

# Bài 4
def is_consecutive_sorted_ans(nums):
    a = sorted(nums)
    return all(y - x == 1 for x, y in zip(a, a[1:]))
assert is_consecutive_sorted_ans([1, 3, 4, 2]) is True
assert is_consecutive_sorted_ans([1, 3]) is False
assert is_consecutive_sorted_ans([4]) is True

# Bài 5
def min_additions_ans(nums):
    return (max(nums) - min(nums) + 1) - len(set(nums))
assert min_additions_ans([1, 3, 4, 2]) == 0
assert min_additions_ans([1, 3]) == 1
assert min_additions_ans([1, 4, 6, 4]) == 3
print("Tất cả đáp án bài tập: OK")

Tất cả đáp án bài tập: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

### Template

```python
from typing import List

class Solution:
    def isConsecutive(self, nums: List[int]) -> bool:
        n = len(nums)
        if len(set(nums)) != n:               # không được trùng
            return False
        return max(nums) - min(nums) == n - 1  # khoảng trải vừa khít
```

### Checklist

- [ ] Hiểu "liên tiếp" = đủ mọi số trong `[min, min + n - 1]`, thứ tự không quan trọng.
- [ ] Kiểm tra **cả hai** điều kiện: không trùng và `max - min == n - 1`.
- [ ] Không dùng tổng hoặc chỉ `max - min` làm điều kiện duy nhất.
- [ ] Mảng 1 phần tử trả `True`.
- [ ] Tránh `v in list` trong vòng lặp với `n` lớn.
- [ ] Đã test số trùng (`[1,1,3]`, `[2,2,3]`) và giá trị lớn tới `10^9`.